In [35]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [36]:
pdfs = list(Path("./raw/").glob("*.pdf"))

In [37]:
import ollama
import fitz

In [19]:
!pip install langchain_ollama

In [38]:
OLLAMA_URL = "http://localhost:11434"
TEXT_EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "llama3.2-vision"

In [39]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_ollama import OllamaEmbeddings
import concurrent
from langchain_community.document_loaders import PyPDFLoader

In [40]:
embeddings = OllamaEmbeddings(
    model="qwen3-embedding",
    base_url="http://localhost:11434"
)

In [41]:
text_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" 
)

In [42]:
from tqdm.auto import tqdm

In [ ]:
def semantic_chunking():
    embeddings = OllamaEmbeddings(
    model="qwen3-embedding",
    base_url="http://localhost:11434"
    )

    text_splitter = SemanticChunker(
    embeddings, 
    breakpoint_threshold_type="percentile" 
    )
    docs = []
    for doc in pdfs:
        loader = PyPDFLoader(doc)

        docs.extend(loader.load())
        
    semantic_chunks = []
    for doc1 in tqdm(docs, desc="Splitting documents"):
        semantic_chunks.extend(text_splitter.split_documents([doc1]))

    return semantic_chunks

semantic_chunking()

In [43]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [44]:
import concurrent.futures
from tqdm import tqdm
# Ensure your Langchain/Ollama imports are here too

def load_single_pdf(pdf_path):
    """Helper function to load a single PDF."""
    loader = PyPDFLoader(pdf_path)
    return loader.load()

def chunk_single_doc(doc, text_splitter):
    """Helper function to chunk a single document."""
    return text_splitter.split_documents([doc])

def semantic_chunking(pdfs):
    embeddings = OllamaEmbeddings(
        model="mxbai-embed-large",
        base_url="http://localhost:11434"
    )

    text_splitter = SemanticChunker(
        embeddings, 
        breakpoint_threshold_type="percentile" 
    )
    pre_splitter = RecursiveCharacterTextSplitter(
        chunk_size=2500,   # characters, not tokens
        chunk_overlap=200
    )
    docs = []
    # 1. Parallelize PDF Loading
    # ThreadPool is great here because file I/O releases the GIL.
    print("Starting parallel PDF loading...")
    with concurrent.futures.ThreadPoolExecutor() as executor:
        # Map applies the function to the iterable concurrently
        results = list(tqdm(executor.map(load_single_pdf, pdfs), total=len(pdfs), desc="Loading PDFs"))
        for res in results:
            docs.extend(res)
    
    docs = pre_splitter.split_documents(docs)
        
    semantic_chunks = []
    
    # 2. Parallelize Semantic Chunking
    # The chunker makes HTTP requests to Ollama, so threads work well.
    # CAUTION: Set max_workers carefully (see note below).
    MAX_WORKERS = 4
    
    print(f"Starting parallel chunking with {MAX_WORKERS} workers...")
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        # Submit tasks to the executor
        futures = [executor.submit(chunk_single_doc, doc, text_splitter) for doc in docs]
        
        # as_completed yields futures as they finish, keeping the progress bar accurate
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(docs), desc="Splitting documents"):
            try:
                semantic_chunks.extend(future.result())
            except Exception as e:
                print(f"Skipping a chunk due to error: {e}")  # Don't crash on one bad doc
    return semantic_chunks

semantic_chunks_parallel = semantic_chunking(pdfs)

Starting parallel PDF loading...


Loading PDFs:   0%|          | 0/18 [00:00<?, ?it/s]

Loading PDFs: 100%|██████████| 18/18 [00:22<00:00,  1.26s/it]


Starting parallel chunking with 4 workers...


Splitting documents:  32%|███▏      | 133/417 [03:57<07:02,  1.49s/it]

Skipping a chunk due to error: the input length exceeds the context length (status code: 400)


Splitting documents:  39%|███▉      | 162/417 [04:38<08:22,  1.97s/it]

Skipping a chunk due to error: the input length exceeds the context length (status code: 400)
Skipping a chunk due to error: the input length exceeds the context length (status code: 400)


Splitting documents: 100%|██████████| 417/417 [10:19<00:00,  1.48s/it]


In [45]:
import chromadb
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction


In [46]:
client = chromadb.PersistentClient(path="chroma_db")

ef = OllamaEmbeddingFunction(
    model_name = "mxbai-embed-large",
    url="http://localhost:11434/api/embeddings"
)

In [28]:
collection = client.get_or_create_collection(name="semantic_texts", embedding_function=ef)

In [49]:
from langchain_ollama import OllamaEmbeddings

embeddings_model = OllamaEmbeddings(model="mxbai-embed-large", base_url="http://localhost:11434")

texts = [doc.page_content for doc in semantic_chunks_parallel]
metadatas = [doc.metadata for doc in semantic_chunks_parallel]

# Embed in batches manually
all_embeddings = []
batch_size = 50
for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
    batch = texts[i : i + batch_size]
    all_embeddings.extend(embeddings_model.embed_documents(batch))

# Add with pre-computed embeddings (no Ollama call needed from ChromaDB)
collection.add(
    ids=[f"chunk_{i}" for i in range(len(texts))],
    documents=texts,
    metadatas=metadatas,
    embeddings=all_embeddings  # pass directly, skips the timeout-prone step
)


Embedding:   0%|          | 0/22 [00:00<?, ?it/s]

Embedding:   0%|          | 0/22 [00:06<?, ?it/s]


ResponseError: the input length exceeds the context length (status code: 400)